# Algorithm 1, step by step, on the EV-charging arm

This notebook runs NeurWIN's training loop (Algorithm 1 of Nakhleh et al., NeurIPS 2021) for **2 mini-batches** and prints the real values at every line, in the paper's notation.

**The arm.** One charging spot (Sec. 5.2, deadline scheduling). Its state is $s = (D, B)$: $D$ = rounds until the vehicle leaves, $B$ = charge it still needs ($D = B = 0$ is an empty spot). Activating the arm ($a = 1$) charges one unit and earns $1 - c = 0.5$; a vehicle that leaves with $B$ units missing costs $F(B) = 0.2\,B^2$.

**What Algorithm 1 learns.** A network $f_\theta(D, B)$ that should be a Whittle index: for an activation cost $\lambda$, activating is worth it exactly when $f_\theta(s) > \lambda$ (*Whittle-accurate*, Def. 4). Training plays $\text{Env}^*(\lambda)$, the version of $\text{Env}(\lambda)$ where the step "activate iff $f_\theta(s) > \lambda$" is replaced by the sigmoid $\sigma_m(f_\theta(s) - \lambda)$, so that the policy has a gradient (Sec. 4.2). By Thm. 1, a $\theta$ that is optimal for every $\text{Env}^*(\lambda)$ gives a Whittle-accurate $f_\theta$ as $m$ grows.

The code is `algorithm1.py` (same folder). It reuses the authors' `fcnn` network and `deadlineSchedulingEnv` and makes the same random draws in the same order, so its result matches the authors' saved checkpoints exactly (last cell).

## Inputs

$\theta$ (the network's initial weights), $\beta \in (0,1)$, learning rate $L$, sensitivity $m$, mini-batch size $R$. These are the paper's values for this arm, and the ones `main.py` uses. The next cell runs Algorithm 1 for 10 episodes (2 mini-batches of $R = 5$) with `trace=True`, which keeps a record of every line.

In [1]:
import sys
from pathlib import Path
for folder in (Path.cwd(), Path.cwd() / "replication", Path.cwd() / "NeurWIN" / "replication"):
    if (folder / "algorithm1.py").exists():
        sys.path.insert(0, str(folder))
        break
else:
    raise FileNotFoundError("algorithm1.py not found: open this notebook from NeurWIN/replication")
import algorithm1

beta, L, m, R, seed = 0.99, 1e-3, 1, 5, 50
EPISODES = 2 * R                                     # 2 mini-batches

theta_plus, trace = algorithm1.train(EPISODES, m=m, L=L, beta=beta, R=R, seed=seed, trace=True)
print(f"Inputs: beta = {beta}, L = {L}, m = {m}, R = {R}, seed = {seed};  {EPISODES} episodes = {len(trace)} mini-batches")
print(f"theta: {sum(p.numel() for p in algorithm1.f_theta_from(theta_plus).parameters())} weights and biases of f_theta (2 -> 16 -> 32 -> 1)")

Inputs: beta = 0.99, L = 0.001, m = 1, R = 5, seed = 50;  10 episodes = 2 mini-batches
theta: 625 weights and biases of f_theta (2 -> 16 -> 32 -> 1)


## L2: pick the activation cost

$$s_0 \sim \text{uniform},\qquad \lambda \leftarrow f_\theta(s_0),\qquad \bar G_b \leftarrow 0$$

**In plain words:** at the start of each mini-batch, imagine a random vehicle $s_0 = (D, B)$ and set the price of charging, $\lambda$, to what the current network says that vehicle is worth. For the whole mini-batch the spot plays $\text{Env}^*(\lambda)$: every unit charged costs $\lambda$. If $f_\theta$ is a good index, charging should be worth that price exactly in states that score above $s_0$.

**Authors' function:** `newMiniBatchReset` (called from `takeAction` at the first round of the mini-batch; draws $B$ from 1..9 first, then $D$ from 1..12).

In [2]:
for mb in trace:
    D0, B0 = mb["s0"]
    print(f"mini-batch b = {mb['b']}:  s0 = (D={D0}, B={B0})   lam = f_theta(s0) = {mb['lam']:+.4f}   G_bar_b <- 0")

mini-batch b = 1:  s0 = (D=1, B=1)   lam = f_theta(s0) = +0.1193   G_bar_b <- 0
mini-batch b = 2:  s0 = (D=1, B=1)   lam = f_theta(s0) = +0.1269   G_bar_b <- 0


## L4: start an episode

$$\text{set the arm to initial state } s_1,\qquad h_e \leftarrow 0$$

**In plain words:** a vehicle arrives at the empty spot and the episode's gradient tally $h_e$ starts at zero. An episode is 300 rounds; whenever a vehicle leaves, the next one may arrive (probability 0.7 per round).

**Authors' function:** `env.reset()`. **Not in Algorithm 1:** the code draws the arrivals (which vehicles come, and when) once per mini-batch, so all $R = 5$ episodes of a mini-batch see the *same* vehicles at the same rounds; only the agent's choices differ.

In [3]:
for mb in trace:
    D1, B1 = mb["s1"]
    print(f"mini-batch b = {mb['b']}:  s1 = (D={D1}, B={B1}) in all {R} episodes   h_e <- 0")

mini-batch b = 1:  s1 = (D=1, B=1) in all 5 episodes   h_e <- 0
mini-batch b = 2:  s1 = (D=3, B=5) in all 5 episodes   h_e <- 0


## L6: choose the action in each round

$$a[t] = 1 \text{ with probability } \sigma_m\big(f_\theta(s[t]) - \lambda\big),\ \text{else } a[t] = 0, \qquad \sigma_m(x) = \frac{1}{1 + e^{-m x}}$$

**In plain words:** each round the spot scores its current vehicle with $f_\theta$. If the score beats the price $\lambda$ it *tends* to charge, but it flips a weighted coin: with $m = 1$ and scores near $\lambda$ the coin is close to 50/50. This randomness is what lets the network learn which choices pay off. With an empty spot $(0, 0)$, activating earns nothing but still pays $\lambda$ (round $t = 1$ below).

**Authors' function:** `takeAction` (`G.choice([1, 0], p=[sigma, 1 - sigma])`). Below: the first 3 rounds of episode 1.

In [4]:
for mb in trace:
    print(f"mini-batch b = {mb['b']}, episode 1   (lam = {mb['lam']:+.4f})")
    print(f"  {'t':>2}  {'s_t = (D, B)':>13}  {'f_theta(s_t)':>12}  {'sigma_m(f - lam)':>16}  {'a_t':>3}")
    for r in mb["first_rounds"]:
        print(f"  {r['t']:>2}  {str(r['s_t']):>13}  {r['f_theta']:>+12.4f}  {r['sigma']:>16.4f}  {r['a_t']:>3}")

mini-batch b = 1, episode 1   (lam = +0.1193)
   t   s_t = (D, B)  f_theta(s_t)  sigma_m(f - lam)  a_t
   0         (1, 1)       +0.1193            0.5000    1
   1         (0, 0)       +0.1108            0.4979    1
   2        (12, 3)       +0.2642            0.5362    1
mini-batch b = 2, episode 1   (lam = +0.1269)
   t   s_t = (D, B)  f_theta(s_t)  sigma_m(f - lam)  a_t
   0         (3, 5)       -0.3261            0.3887    1
   1         (2, 4)       -0.2251            0.4129    0
   2         (1, 4)       -0.2325            0.4111    0


## L7–L10: add up the gradient of the choice

$$\text{if } a[t] = 1:\ h_e \leftarrow h_e + \nabla_\theta \ln \sigma_m\big(f_\theta(s[t]) - \lambda\big) \qquad \text{else: } h_e \leftarrow h_e + \nabla_\theta \ln\big(1 - \sigma_m(f_\theta(s[t]) - \lambda)\big)$$

**In plain words:** for every round, $h_e$ records which way to move $\theta$ to make the choice that was just made *more likely*. Both charging and not charging add a term: every round counts. Later (L16) these directions are pushed forward or backward depending on whether the episode went well.

**Authors' function:** `takeAction` (`logProb.backward()` adds the round's gradient into `.grad`), then `_saveEpisodeGradients` copies the episode's total out. Below: the length $\lVert h_e \rVert$ of each episode's tally (625 numbers, one per weight).

In [5]:
for mb in trace:
    print(f"mini-batch b = {mb['b']}:  |h_e| for e = 1..{R}:  " + "  ".join(f"{x:7.2f}" for x in mb["h_e_size"]))

mini-batch b = 1:  |h_e| for e = 1..5:    27.35    31.26    28.31    26.87    27.39
mini-batch b = 2:  |h_e| for e = 1..5:    48.95    29.67    23.17    25.53    27.39


## L12: score the episode

$$G_e \leftarrow \sum_t \beta^t \big(r[t] - \lambda\, a[t]\big) \quad \text{(empirical discounted net reward)}$$

**In plain words:** add up what the spot earned over the 300 rounds: $+0.5$ per unit charged, $-0.2B^2$ when a vehicle leaves short, minus the price $\lambda$ for every round it charged; later rounds count a little less ($\beta = 0.99$).

**Authors' function:** `learn()` subtracts the cost (`reward -= self.cost`), `_discountRewards` discounts and sums. (It returns $-G_e$; the code keeps the minus sign and undoes it with Adam's descent step in L16.)

In [6]:
for mb in trace:
    r0 = mb["first_rounds"][0]
    print(f"mini-batch b = {mb['b']}:  first round of episode 1: r[0] = {r0['r_t']:+.2f}, a[0] = {r0['a_t']}, "
          f"net reward r - lam*a = {r0['net']:+.4f}")
    for e, (G_e, n) in enumerate(zip(mb["G_e"], mb["charged"]), 1):
        print(f"    episode e = {e}:  G_e = {G_e:8.3f}   rounds charged = {n:3d} / 300")

mini-batch b = 1:  first round of episode 1: r[0] = +0.50, a[0] = 1, net reward r - lam*a = +0.3807
    episode e = 1:  G_e =  -15.651   rounds charged = 136 / 300
    episode e = 2:  G_e =  -13.954   rounds charged = 143 / 300
    episode e = 3:  G_e =  -15.781   rounds charged = 125 / 300
    episode e = 4:  G_e =  -20.910   rounds charged = 130 / 300
    episode e = 5:  G_e =  -17.807   rounds charged = 125 / 300
mini-batch b = 2:  first round of episode 1: r[0] = +0.50, a[0] = 1, net reward r - lam*a = +0.3731
    episode e = 1:  G_e =  -36.628   rounds charged = 124 / 300
    episode e = 2:  G_e =  -36.723   rounds charged = 123 / 300
    episode e = 3:  G_e =  -34.738   rounds charged = 126 / 300
    episode e = 4:  G_e =  -26.688   rounds charged = 145 / 300
    episode e = 5:  G_e =  -39.117   rounds charged = 139 / 300


## L13: the bootstrapped baseline

$$\bar G_b \leftarrow \bar G_b + G_e / R$$

**In plain words:** keep a running average of the $R$ episode scores. It is the "typical" result for this mini-batch, so an episode is judged by whether it did better or worse than its siblings, not by its raw score (which is negative mostly because of departure penalties).

**Authors' function:** `_performBatchStep` (`meanBatchReward`).

In [7]:
for mb in trace:
    print(f"mini-batch b = {mb['b']}:  G_bar_b = mean of the {R} G_e = {mb['G_bar_b']:.3f}")

mini-batch b = 1:  G_bar_b = mean of the 5 G_e = -16.821
mini-batch b = 2:  G_bar_b = mean of the 5 G_e = -34.779


## L15: the learning rate

$$L_b \leftarrow \text{learning rate in mini-batch } b$$

**In plain words:** how big a step to take. Here $L_b = L = 0.001$ in every mini-batch, used by the Adam optimizer, which also rescales each weight's step.

**Authors' function:** `torch.optim.Adam(..., lr=L)` built in `NEURWIN.__init__`.

In [8]:
for mb in trace:
    print(f"mini-batch b = {mb['b']}:  L_b = {L}")

mini-batch b = 1:  L_b = 0.001
mini-batch b = 2:  L_b = 0.001


## L16: gradient ascent

$$\theta \leftarrow \theta + L_b \sum_{e=1}^{R} \big(G_e - \bar G_b\big)\, h_e$$

**In plain words:** episodes that beat the baseline (positive *offset net reward* $G_e - \bar G_b$) make their choices more likely; episodes that fell short make theirs less likely. The result is a new $\theta$, hence new scores $f_\theta(D, B)$.

**Authors' function:** `_performBatchStep`. Adam *minimizes*, so the code stores $-\sum_e (G_e - \bar G_b) h_e$ in `.grad` and calls `optimizer.step()`: stepping down that is stepping up the paper's direction, which is gradient ascent. Below: the offsets, how far $\theta$ moved, and how the scores of three vehicles changed.

In [9]:
for mb in trace:
    print(f"mini-batch b = {mb['b']}:  offset net rewards G_e - G_bar_b: " + " ".join(f"{x:+.2f}" for x in mb["offsets"]))
    print(f"    theta moved |theta_new - theta_old| = {mb['theta_change']:.5f}")
    for w in mb["watch"]:
        print(f"    f_theta(D={w['D']}, B={w['B']}): {w['before']:+.4f} -> {w['after']:+.4f}")

mini-batch b = 1:  offset net rewards G_e - G_bar_b: +1.17 +2.87 +1.04 -4.09 -0.99
    theta moved |theta_new - theta_old| = 0.01874
    f_theta(D=2, B=4): -0.2497 -> -0.2251
    f_theta(D=1, B=3): -0.1365 -> -0.1180
    f_theta(D=6, B=2): +0.1771 -> +0.1880
mini-batch b = 2:  offset net rewards G_e - G_bar_b: -1.85 -1.94 +0.04 +8.09 -4.34
    theta moved |theta_new - theta_old| = 0.01635
    f_theta(D=2, B=4): -0.2251 -> -0.2013
    f_theta(D=1, B=3): -0.1180 -> -0.1002
    f_theta(D=6, B=2): +0.1880 -> +0.2046


## Check against the expected values

The values worked out for this replication. Any mismatch is printed in capitals.

In [10]:
expected = {
    1: dict(s0=(1, 1), lam=0.1193, G_bar_b=-16.821, offsets=[1.17, 2.87, 1.04, -4.09, -0.99], theta_change=0.01874),
    2: dict(lam=0.1269, G_bar_b=-34.779, offsets=[-1.85, -1.94, 0.04, 8.09, -4.34], theta_change=0.01635),
}
all_ok = True
for mb in trace:
    exp = expected[mb["b"]]
    got = dict(s0=mb["s0"], lam=round(mb["lam"], 4), G_bar_b=round(mb["G_bar_b"], 3),
               offsets=[round(x, 2) for x in mb["offsets"]], theta_change=round(mb["theta_change"], 5))
    for key, want in exp.items():
        ok = got[key] == want
        all_ok &= ok
        print(f"mini-batch {mb['b']}  {key:<13} expected {str(want):<34} got {str(got[key]):<34} {'ok' if ok else 'MISMATCH'}")
print("\nALL MATCH" if all_ok else "\nMISMATCH FOUND")

mini-batch 1  s0            expected (1, 1)                             got (1, 1)                             ok
mini-batch 1  lam           expected 0.1193                             got 0.1193                             ok
mini-batch 1  G_bar_b       expected -16.821                            got -16.821                            ok
mini-batch 1  offsets       expected [1.17, 2.87, 1.04, -4.09, -0.99]   got [1.17, 2.87, 1.04, -4.09, -0.99]   ok
mini-batch 1  theta_change  expected 0.01874                            got 0.01874                            ok
mini-batch 2  lam           expected 0.1269                             got 0.1269                             ok
mini-batch 2  G_bar_b       expected -34.779                            got -34.779                            ok
mini-batch 2  offsets       expected [-1.85, -1.94, 0.04, 8.09, -4.34]  got [-1.85, -1.94, 0.04, 8.09, -4.34]  ok
mini-batch 2  theta_change  expected 0.01635                            got 0.01635     

## Output after 2 mini-batches: the checkpoint check

After 10 episodes, $\theta$ should be exactly the authors' saved checkpoint `trainResults/neurwin/deadline_env/seed_50_lr_0.001_batchSize_5_trainedNumEpisodes_10`. (Running `python algorithm1.py` also checks the full 2,000 episodes against the final checkpoint.)

In [11]:
diff = algorithm1.max_difference(theta_plus, EPISODES)
print(f"max |theta after {len(trace)} mini-batches - authors' checkpoint {EPISODES}| over all 625 numbers = {diff}")

max |theta after 2 mini-batches - authors' checkpoint 10| over all 625 numbers = 0.0
